# Notebook 05c — SHAP Stability Tests on Canonical Shared Samples

**Purpose:** Measure how robust SHAP explanations are to small input perturbations. Generates Jaccard top-10 stability scores per sample, which feed Notebook 07c's SCTS-v2 component c₂.

**Why canonical samples:** Same canonical 1000 per-dataset indices used in Notebooks 04c and 06v3. Ensures per-sample stability scores align with SHAP outputs and cross-architecture disagreement scores for downstream SCTS-v2 integration.

**Methodology (identical to v1 `05_stability_tests.ipynb`):**
- Three perturbation types: Gaussian noise, FGSM (single-step gradient sign), PGD (10-step projected)
- `EPSILON = 0.05` (5% of unit variance), `ALPHA = 0.005` (PGD step), `PGD_STEPS = 10`
- Transfer attack pattern: FGSM/PGD computed against the DNN-CW model of each dataset, then applied to all 6 models (trees + DNNs). This is the standard cross-model adversarial XAI approach (Arreche 2024).
- Per-sample Jaccard top-10 between clean SHAP (from 04c) and perturbed SHAP (this notebook).

**Stability metrics:**
- **Jaccard top-10** — per-sample fraction of top-10 features overlapping (clean vs perturbed). Range [0,1].
- **Lipschitz median** — `‖ΔSHAP‖ / ‖ΔX‖` median across samples (robust scalar).
- F-Fidelity dropped here (not needed for SCTS-v2 c₂; adds complexity without direct paper claim support).

**Scope:** 18 models × 3 perturbations = 54 SHAP recomputations.

**Time estimate:** ~2-3 hours on T4 GPU. Trees are slow on adversarial input (TreeExplainer re-fits), DNN-GradientExplainer is fast.

**Resumability:** Each (model, perturbation) combo saves independently; skip on re-run if already done. Designed for Drive-disconnect recovery.

**Outputs:**
- `shap_values/{ds}/perturbations/{type}_X.npy` — perturbed input arrays (per dataset)
- `shap_values/{ds}/perturbed_shap/{model}_{type}_shap.npy` — perturbed SHAP per (model, perturbation)
- `results/tables/stability_v2.csv` — aggregate metrics per (model, perturbation)
- `results/tables/stability_v2_per_sample_jaccard.csv` — per-sample Jaccard for SCTS-v2 input
- `results/tables/stability_v2_summary.json` — aggregate + per-architecture stats

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json
import joblib
import time
import traceback
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import shap

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch device: {DEVICE}')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Perturbation parameters (verbatim from 05_stability_tests.ipynb)
EPSILON = 0.05
ALPHA = 0.005
PGD_STEPS = 10

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
ARCHITECTURES = ['rf', 'xgb', 'dnn']
VARIANTS = ['5class_cw', '5class_smote']
PERTURBATIONS = ['gaussian', 'fgsm', 'pgd']

# DNN used to generate FGSM/PGD perturbations (one per dataset)
ATTACK_SOURCE_MODEL = 'dnn_5class_cw'

print(f'Scope: {len(DATASETS)} datasets × 6 models × {len(PERTURBATIONS)} perturbations = {len(DATASETS) * 6 * len(PERTURBATIONS)} SHAP recomputations')

PyTorch device: cuda
Scope: 3 datasets × 6 models × 3 perturbations = 54 SHAP recomputations


## 2. Model helpers (verbatim from 04c)

In [3]:
def find_tree_model_path(dataset, model_name):
    base = Path(REPO) / 'models' / dataset
    for ext in ['.pkl', '.joblib']:
        p = base / f'{model_name}{ext}'
        if p.exists():
            return p
    raise FileNotFoundError(f'No tree model file for {model_name} in {base}')

def find_dnn_path(dataset, model_name):
    p = Path(REPO) / 'models' / dataset / f'{model_name}.pt'
    if p.exists():
        return p
    raise FileNotFoundError(f'No DNN file at {p}')

class DNN(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(256, 128, 64, 32), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class DNNWithSoftmax(nn.Module):
    def __init__(self, dnn):
        super().__init__()
        self.dnn = dnn
    def forward(self, x):
        return torch.softmax(self.dnn(x), dim=-1)

def load_pytorch_dnn(path, return_raw=False):
    """Load DNN. return_raw=True returns the bare model (for FGSM/PGD gradients on logits).
    return_raw=False returns the softmax-wrapped version (for SHAP).
    """
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        in_dim = ckpt['in_dim']
        n_classes = ckpt['n_classes']
        hidden = tuple(ckpt['hidden'])
        dropout = ckpt['dropout']
        state_dict = ckpt['state_dict']
    else:
        state_dict = ckpt
        in_dim = state_dict['net.0.weight'].shape[1]
        n_classes = state_dict['net.16.weight'].shape[0]
        hidden = (256, 128, 64, 32)
        dropout = 0.3
    model = DNN(in_dim=in_dim, n_classes=n_classes, hidden=hidden, dropout=dropout)
    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()
    if return_raw:
        return model
    return DNNWithSoftmax(model).to(DEVICE).eval()

print('✓ Model helpers loaded')

✓ Model helpers loaded


## 3. Perturbation generators

In [4]:
def gaussian_perturbation(X, epsilon, seed):
    """Add N(0, epsilon) noise per feature."""
    rng = np.random.RandomState(seed)
    return (X + rng.normal(0, epsilon, X.shape)).astype(np.float32)

def fgsm_attack(model_raw, X, y, epsilon):
    """Single-step FGSM. model_raw should output LOGITS (not softmax)."""
    model_raw.eval()
    X_t = torch.tensor(X, dtype=torch.float32, requires_grad=True, device=DEVICE)
    y_t = torch.tensor(y, dtype=torch.long, device=DEVICE)
    logits = model_raw(X_t)
    loss = F.cross_entropy(logits, y_t)
    grad = torch.autograd.grad(loss, X_t)[0]
    X_pert = X_t + epsilon * grad.sign()
    return X_pert.detach().cpu().numpy().astype(np.float32)

def pgd_attack(model_raw, X, y, epsilon, alpha, steps, seed=None):
    """Multi-step PGD with random init. model_raw should output LOGITS."""
    model_raw.eval()
    X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    y_t = torch.tensor(y, dtype=torch.long, device=DEVICE)
    if seed is not None:
        torch.manual_seed(seed)
    delta = torch.empty_like(X_t).uniform_(-epsilon, epsilon).to(DEVICE)
    X_adv = X_t + delta
    for _ in range(steps):
        X_adv = X_adv.detach().requires_grad_(True)
        logits = model_raw(X_adv)
        loss = F.cross_entropy(logits, y_t)
        grad = torch.autograd.grad(loss, X_adv)[0]
        X_adv = X_adv + alpha * grad.sign()
        X_adv = torch.max(torch.min(X_adv, X_t + epsilon), X_t - epsilon)
    return X_adv.detach().cpu().numpy().astype(np.float32)

print('✓ Perturbation generators loaded')

✓ Perturbation generators loaded


## 4. Generate perturbed inputs per dataset (saves them once, reuses across all models)

In [5]:
for ds in DATASETS:
    print(f'\n=== {ds} ===')

    # Load canonical eval indices and corresponding samples
    canonical_path = Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy'
    eval_idx = np.load(canonical_path)

    X_test = np.load(f'{REPO}/data/processed/{ds}/X_test.npy').astype(np.float32)
    y_test = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')

    X_canonical = X_test[eval_idx]
    y_canonical = y_test[eval_idx]

    # Setup output directory
    pert_dir = Path(REPO) / 'shap_values' / ds / 'perturbations'
    pert_dir.mkdir(parents=True, exist_ok=True)

    # 1. Gaussian (model-agnostic, just noise)
    gaussian_path = pert_dir / 'gaussian_X.npy'
    if gaussian_path.exists():
        print(f'  Gaussian: cached at {gaussian_path.name}')
    else:
        X_gauss = gaussian_perturbation(X_canonical, EPSILON, seed=SEED)
        np.save(gaussian_path, X_gauss)
        delta = np.linalg.norm(X_gauss - X_canonical, axis=1).mean()
        print(f'  ✓ Gaussian saved: ‖δ‖_mean = {delta:.3f}')

    # 2. FGSM (generated from DNN-CW)
    fgsm_path = pert_dir / 'fgsm_X.npy'
    if fgsm_path.exists():
        print(f'  FGSM: cached at {fgsm_path.name}')
    else:
        dnn_raw = load_pytorch_dnn(find_dnn_path(ds, ATTACK_SOURCE_MODEL), return_raw=True)
        X_fgsm = fgsm_attack(dnn_raw, X_canonical, y_canonical, EPSILON)
        np.save(fgsm_path, X_fgsm)
        delta = np.linalg.norm(X_fgsm - X_canonical, axis=1).mean()
        print(f'  ✓ FGSM saved (source={ATTACK_SOURCE_MODEL}): ‖δ‖_mean = {delta:.3f}')
        del dnn_raw
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    # 3. PGD (generated from DNN-CW)
    pgd_path = pert_dir / 'pgd_X.npy'
    if pgd_path.exists():
        print(f'  PGD: cached at {pgd_path.name}')
    else:
        dnn_raw = load_pytorch_dnn(find_dnn_path(ds, ATTACK_SOURCE_MODEL), return_raw=True)
        X_pgd = pgd_attack(dnn_raw, X_canonical, y_canonical, EPSILON, ALPHA, PGD_STEPS, seed=SEED)
        np.save(pgd_path, X_pgd)
        delta = np.linalg.norm(X_pgd - X_canonical, axis=1).mean()
        print(f'  ✓ PGD saved (source={ATTACK_SOURCE_MODEL}): ‖δ‖_mean = {delta:.3f}')
        del dnn_raw
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()


=== nsl_kdd_v2 ===
  ✓ Gaussian saved: ‖δ‖_mean = 0.552
  ✓ FGSM saved (source=dnn_5class_cw): ‖δ‖_mean = 0.551
  ✓ PGD saved (source=dnn_5class_cw): ‖δ‖_mean = 0.443

=== unsw_nb15_v2 ===
  ✓ Gaussian saved: ‖δ‖_mean = 0.695
  ✓ FGSM saved (source=dnn_5class_cw): ‖δ‖_mean = 0.696
  ✓ PGD saved (source=dnn_5class_cw): ‖δ‖_mean = 0.530

=== cic_ids2017_v2 ===
  ✓ Gaussian saved: ‖δ‖_mean = 0.441
  ✓ FGSM saved (source=dnn_5class_cw): ‖δ‖_mean = 0.442
  ✓ PGD saved (source=dnn_5class_cw): ‖δ‖_mean = 0.355


## 5. Compute SHAP on perturbed inputs for all 18 models × 3 perturbations

In [6]:
def compute_perturbed_shap(dataset, model_name, perturbation, X_pert, bg_idx, bg_X):
    """Recompute SHAP on perturbed input. Identical methodology to 04c."""
    out_path = Path(REPO) / 'shap_values' / dataset / 'perturbed_shap' / f'{model_name}_{perturbation}_shap.npy'
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists():
        # Resumability: skip if already done
        return out_path, 0.0, True  # cached

    t0 = time.time()
    n_classes = 5
    n_features = X_pert.shape[1]
    model_type = 'rf' if 'rf' in model_name else ('xgb' if 'xgb' in model_name else 'dnn')

    if model_type in ('rf', 'xgb'):
        path = find_tree_model_path(dataset, model_name)
        model = joblib.load(path)
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_pert, check_additivity=False)

        if isinstance(shap_values, list):
            shap_values = np.stack(shap_values, axis=-1)
        elif shap_values.ndim == 3:
            if shap_values.shape[0] == n_classes:
                shap_values = np.transpose(shap_values, (1, 2, 0))
            elif shap_values.shape[1] == n_classes:
                shap_values = np.transpose(shap_values, (0, 2, 1))

    elif model_type == 'dnn':
        path = find_dnn_path(dataset, model_name)
        wrapped_model = load_pytorch_dnn(path, return_raw=False)

        bg_tensor = torch.from_numpy(bg_X).to(DEVICE)
        eval_tensor = torch.from_numpy(X_pert).to(DEVICE)

        explainer = shap.GradientExplainer(wrapped_model, bg_tensor)
        shap_values = explainer.shap_values(eval_tensor)

        if isinstance(shap_values, list):
            shap_values = np.stack(shap_values, axis=-1)
        elif shap_values.ndim == 3:
            if shap_values.shape[0] == n_classes:
                shap_values = np.transpose(shap_values, (1, 2, 0))
            elif shap_values.shape[1] == n_classes:
                shap_values = np.transpose(shap_values, (0, 2, 1))

        del wrapped_model, bg_tensor, eval_tensor
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    # Final shape sanity
    expected = (X_pert.shape[0], n_features, n_classes)
    if shap_values.shape != expected:
        if shap_values.shape == (n_classes, X_pert.shape[0], n_features):
            shap_values = np.transpose(shap_values, (1, 2, 0))
        elif shap_values.shape == (X_pert.shape[0], n_classes, n_features):
            shap_values = np.transpose(shap_values, (0, 2, 1))

    np.save(out_path, shap_values)
    return out_path, time.time() - t0, False  # computed fresh

# Main loop
MODELS_PER_DATASET = [f'{a}_{v}' for v in VARIANTS for a in ARCHITECTURES]
perturbed_results = []
errors = []
t_overall = time.time()

print(f'\n{"="*70}')
print(f'Perturbed SHAP computation — {datetime.now().strftime("%H:%M:%S")}')
print(f'{"="*70}\n')

for ds in DATASETS:
    pert_dir = Path(REPO) / 'shap_values' / ds / 'perturbations'
    canonical_eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    canonical_bg_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_bg_idx.npy')

    X_calib = np.load(f'{REPO}/data/processed/{ds}/X_calib.npy').astype(np.float32)
    bg_X = X_calib[canonical_bg_idx]

    pert_arrays = {p: np.load(pert_dir / f'{p}_X.npy') for p in PERTURBATIONS}

    print(f'\n=== {ds} ===')

    for model_name in MODELS_PER_DATASET:
        for perturbation in PERTURBATIONS:
            X_pert = pert_arrays[perturbation]

            print(f'  ▶  {model_name:<22} {perturbation:<10} ', end='', flush=True)
            try:
                out_path, elapsed, was_cached = compute_perturbed_shap(
                    ds, model_name, perturbation, X_pert,
                    canonical_bg_idx, bg_X
                )
                if was_cached:
                    print(f'(cached)')
                else:
                    print(f'{elapsed:.1f}s')
                perturbed_results.append({
                    'dataset': ds, 'model': model_name, 'perturbation': perturbation,
                    'path': str(out_path.relative_to(REPO)),
                    'time_seconds': round(elapsed, 1),
                    'cached': was_cached,
                })
            except Exception as e:
                print(f'❌ FAILED: {type(e).__name__}: {e}')
                errors.append({
                    'dataset': ds, 'model': model_name, 'perturbation': perturbation,
                    'error': str(e), 'tb': traceback.format_exc(),
                })

elapsed_total = (time.time() - t_overall) / 60
print(f'\nTotal time: {elapsed_total:.1f} min')
print(f'Completed: {len(perturbed_results)}, Failed: {len(errors)}')


Perturbed SHAP computation — 19:28:16


=== nsl_kdd_v2 ===
  ▶  rf_5class_cw           gaussian   130.6s
  ▶  rf_5class_cw           fgsm       129.2s
  ▶  rf_5class_cw           pgd        125.7s
  ▶  xgb_5class_cw          gaussian   10.2s
  ▶  xgb_5class_cw          fgsm       8.5s
  ▶  xgb_5class_cw          pgd        9.6s
  ▶  dnn_5class_cw          gaussian   319.7s
  ▶  dnn_5class_cw          fgsm       319.6s
  ▶  dnn_5class_cw          pgd        320.1s
  ▶  rf_5class_smote        gaussian   156.6s
  ▶  rf_5class_smote        fgsm       157.1s
  ▶  rf_5class_smote        pgd        158.0s
  ▶  xgb_5class_smote       gaussian   13.4s
  ▶  xgb_5class_smote       fgsm       12.2s
  ▶  xgb_5class_smote       pgd        12.5s
  ▶  dnn_5class_smote       gaussian   321.5s
  ▶  dnn_5class_smote       fgsm       322.5s
  ▶  dnn_5class_smote       pgd        323.7s

=== unsw_nb15_v2 ===
  ▶  rf_5class_cw           gaussian   1287.1s
  ▶  rf_5class_cw           fgsm       1279.3s
  ▶ 

## 6. Compute stability metrics: per-sample Jaccard top-10 + Lipschitz median

In [7]:
def jaccard_topk_per_sample(shap_orig, shap_pert, k=10):
    """Per-sample Jaccard top-k. Returns array (n_samples,) of Jaccard scores in [0,1].
    Uses sum(|SHAP|) across classes as per-sample importance, identical to 05_stability_tests."""
    if shap_orig.ndim == 3:
        imp_a = np.abs(shap_orig).sum(axis=-1)
        imp_b = np.abs(shap_pert).sum(axis=-1)
    else:
        imp_a = np.abs(shap_orig)
        imp_b = np.abs(shap_pert)

    n_samples = imp_a.shape[0]
    jaccards = np.zeros(n_samples)
    for i in range(n_samples):
        top_a = set(np.argsort(-imp_a[i])[:k])
        top_b = set(np.argsort(-imp_b[i])[:k])
        union = len(top_a | top_b)
        jaccards[i] = len(top_a & top_b) / union if union > 0 else 1.0
    return jaccards

def lipschitz_estimate(shap_orig, shap_pert, X_orig, X_pert):
    """Per-sample Lipschitz: ‖ΔSHAP‖_F / ‖ΔX‖_2. Returns median."""
    if shap_orig.ndim == 3:
        shap_diff = (shap_pert - shap_orig).reshape(shap_orig.shape[0], -1)
    else:
        shap_diff = shap_pert - shap_orig
    shap_norms = np.linalg.norm(shap_diff, axis=1)
    x_norms = np.linalg.norm(X_pert - X_orig, axis=1)
    mask = x_norms > 1e-8
    if mask.sum() == 0:
        return float('nan')
    ratios = shap_norms[mask] / x_norms[mask]
    return float(np.median(ratios))

# Compute stability metrics
stability_rows = []
per_sample_jaccard_rows = []

print(f'\nComputing stability metrics...')
for ds in DATASETS:
    pert_dir = Path(REPO) / 'shap_values' / ds / 'perturbations'
    eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    X_test = np.load(f'{REPO}/data/processed/{ds}/X_test.npy').astype(np.float32)
    y_test = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')
    X_canonical = X_test[eval_idx]
    y_canonical = y_test[eval_idx]

    pert_arrays = {p: np.load(pert_dir / f'{p}_X.npy') for p in PERTURBATIONS}

    for model_name in MODELS_PER_DATASET:
        # Load original (clean) SHAP from 04c
        orig_shap_path = Path(REPO) / 'shap_values' / ds / f'{model_name}_shap_shared.npy'
        if not orig_shap_path.exists():
            continue
        shap_orig = np.load(orig_shap_path)

        for perturbation in PERTURBATIONS:
            pert_shap_path = Path(REPO) / 'shap_values' / ds / 'perturbed_shap' / f'{model_name}_{perturbation}_shap.npy'
            if not pert_shap_path.exists():
                continue
            shap_pert = np.load(pert_shap_path)

            X_pert = pert_arrays[perturbation]

            # Per-sample Jaccard
            jaccards = jaccard_topk_per_sample(shap_orig, shap_pert, k=10)

            # Lipschitz median
            lip = lipschitz_estimate(shap_orig, shap_pert, X_canonical, X_pert)

            # Aggregate row
            stability_rows.append({
                'dataset': ds, 'model': model_name, 'perturbation': perturbation,
                'jaccard_top10_mean': float(np.mean(jaccards)),
                'jaccard_top10_median': float(np.median(jaccards)),
                'jaccard_top10_q25': float(np.quantile(jaccards, 0.25)),
                'jaccard_top10_q75': float(np.quantile(jaccards, 0.75)),
                'lipschitz_median': lip,
            })

            # Per-sample rows for SCTS-v2 input
            for i, j_val in enumerate(jaccards):
                per_sample_jaccard_rows.append({
                    'dataset': ds, 'model': model_name, 'perturbation': perturbation,
                    'sample_position': i, 'true_class': int(y_canonical[i]),
                    'jaccard_top10': float(j_val),
                })

df_stab = pd.DataFrame(stability_rows)
df_per_sample = pd.DataFrame(per_sample_jaccard_rows)

out_dir = Path(REPO) / 'results' / 'tables'
out_dir.mkdir(parents=True, exist_ok=True)
df_stab.to_csv(out_dir / 'stability_v2.csv', index=False)
df_per_sample.to_csv(out_dir / 'stability_v2_per_sample_jaccard.csv', index=False)

print(f'✓ Saved: stability_v2.csv ({len(df_stab)} rows)')
print(f'✓ Saved: stability_v2_per_sample_jaccard.csv ({len(df_per_sample)} rows)')


Computing stability metrics...
✓ Saved: stability_v2.csv (54 rows)
✓ Saved: stability_v2_per_sample_jaccard.csv (54000 rows)


## 7. Summary stats + test DNN-stability-advantage prediction (v7/v8)

In [8]:
print('=' * 70)
print('STABILITY SUMMARY (mean Jaccard top-10 per perturbation × architecture)')
print('=' * 70)

# Add architecture column
df_stab['architecture'] = df_stab['model'].apply(
    lambda m: 'rf' if 'rf' in m else ('xgb' if 'xgb' in m else 'dnn')
)

# Mean Jaccard per (perturbation, architecture) across all datasets and variants
agg = df_stab.groupby(['perturbation', 'architecture'])['jaccard_top10_mean'].agg(['mean', 'std', 'count'])
print(agg.round(3))

# Per-dataset breakdown
print('\n' + '=' * 70)
print('PER-DATASET MEAN JACCARD (averaged across both variants)')
print('=' * 70)
for ds in DATASETS:
    sub = df_stab[df_stab['dataset'] == ds]
    pivot = sub.groupby(['perturbation', 'architecture'])['jaccard_top10_mean'].mean().unstack()
    print(f'\n{ds}:')
    print(pivot.round(3))

# PREDICTION TEST: DNN stability advantage
# v7/v8 claim: DNN explanations are more stable than tree explanations under perturbation
# Test: per (dataset, variant, perturbation), is DNN Jaccard > both RF and XGB Jaccard?
print('\n' + '=' * 70)
print('PREDICTION: DNN stability > tree stability under adversarial perturbation')
print('=' * 70)

pred_dnn_advantage = []
for ds in DATASETS:
    for variant in VARIANTS:
        for perturbation in PERTURBATIONS:
            sub = df_stab[
                (df_stab['dataset'] == ds) &
                (df_stab['model'].str.contains(variant)) &
                (df_stab['perturbation'] == perturbation)
            ]
            if len(sub) != 3:
                continue

            arch_jaccard = sub.set_index('architecture')['jaccard_top10_mean'].to_dict()
            dnn_j = arch_jaccard.get('dnn', None)
            rf_j = arch_jaccard.get('rf', None)
            xgb_j = arch_jaccard.get('xgb', None)

            if None in (dnn_j, rf_j, xgb_j):
                continue

            supports = bool(dnn_j > max(rf_j, xgb_j))
            pred_dnn_advantage.append({
                'dataset': ds, 'variant': variant, 'perturbation': perturbation,
                'dnn_jaccard': round(dnn_j, 3),
                'rf_jaccard': round(rf_j, 3),
                'xgb_jaccard': round(xgb_j, 3),
                'supports_dnn_advantage': supports,
            })
            print(f'  {ds:<18} {variant:<14} {perturbation:<10} '
                  f'dnn={dnn_j:.3f}, rf={rf_j:.3f}, xgb={xgb_j:.3f}, dnn_advantage={supports}')

n_supports = sum(1 for r in pred_dnn_advantage if r['supports_dnn_advantage'])
n_total = len(pred_dnn_advantage)
print(f'\nDNN stability advantage supported in {n_supports} of {n_total} combos')

STABILITY SUMMARY (mean Jaccard top-10 per perturbation × architecture)
                            mean    std  count
perturbation architecture                     
fgsm         dnn           0.574  0.049      6
             rf            0.482  0.135      6
             xgb           0.556  0.089      6
gaussian     dnn           0.602  0.055      6
             rf            0.515  0.130      6
             xgb           0.575  0.084      6
pgd          dnn           0.590  0.050      6
             rf            0.508  0.139      6
             xgb           0.566  0.085      6

PER-DATASET MEAN JACCARD (averaged across both variants)

nsl_kdd_v2:
architecture    dnn     rf    xgb
perturbation                     
fgsm          0.600  0.618  0.655
gaussian      0.615  0.651  0.672
pgd           0.608  0.643  0.664

unsw_nb15_v2:
architecture    dnn     rf    xgb
perturbation                     
fgsm          0.559  0.496  0.479
gaussian      0.596  0.518  0.503
pgd           0.583

## 8. Save summary JSON

In [9]:
import numpy as np

def to_json_safe(obj):
    """Recursively convert numpy types to Python natives for JSON."""
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_json_safe(x) for x in obj]
    elif isinstance(obj, (np.bool_, np.generic)):
        return obj.item()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

summary = {
    'timestamp': datetime.now().isoformat(),
    'method': 'canonical_shared_samples_per_dataset',
    'n_datasets': len(DATASETS),
    'n_models': len(MODELS_PER_DATASET) * len(DATASETS),
    'n_perturbations': len(PERTURBATIONS),
    'perturbation_params': {
        'epsilon': EPSILON,
        'pgd_alpha': ALPHA,
        'pgd_steps': PGD_STEPS,
        'attack_source_model': ATTACK_SOURCE_MODEL,
    },
    'mean_jaccard_by_arch_perturbation': df_stab.groupby(['perturbation', 'architecture'])['jaccard_top10_mean'].agg(['mean', 'std']).round(4).to_dict(),
    'prediction_dnn_stability_advantage': {
        'supported_count': sum(1 for r in pred_dnn_advantage if r['supports_dnn_advantage']),
        'total_count': len(pred_dnn_advantage),
        'details': pred_dnn_advantage,
    },
    'overall_stats': {
        'min_jaccard': float(df_stab['jaccard_top10_mean'].min()),
        'max_jaccard': float(df_stab['jaccard_top10_mean'].max()),
        'mean_jaccard': float(df_stab['jaccard_top10_mean'].mean()),
        'median_jaccard': float(df_stab['jaccard_top10_mean'].median()),
    },
}

summary_safe = to_json_safe(summary)
out_path = Path(REPO) / 'results' / 'tables' / 'stability_v2_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary_safe, f, indent=2, default=str)
print(f'✓ Saved: stability_v2_summary.json')

TypeError: keys must be str, int, float, bool or None, not tuple

## 9. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/05c_stability_canonical.ipynb
!git add results/tables/stability_v2.csv
!git add results/tables/stability_v2_per_sample_jaccard.csv
!git add results/tables/stability_v2_summary.json
!git status --short
!git commit -m 'Notebook 05c: SHAP stability on canonical samples (54 perturbed SHAP arrays, Jaccard top-10 + Lipschitz median, 18 models x 3 perturbations)'
!git push origin main